# 📖 Notebook 4: Trip Lifecycle Management

A ride goes through many states: requested → matching → assigned → accepted → en_route → arrived → in_progress → completed. At each step, multiple things can go wrong: a driver might not respond, two services might try to assign the same driver, or a rider might cancel mid-match.

This notebook covers:
- The **ride state machine** — valid transitions between states
- **Distributed locking with Redis** — preventing two rides from claiming the same driver
- **TTL-based locks** — automatically releasing locks if a driver doesn't respond
- **The complete ride flow** from request to completion

## Learning Objectives

By the end of this notebook, you'll understand:
- How to model a ride as a state machine with strict transition rules
- Why distributed locks are needed (and what breaks without them)
- How Redis SET NX EX implements a lock with automatic expiry
- How to handle driver timeouts and move to the next candidate

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/uber
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import json

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "uber_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🔄 The Ride State Machine

A ride is not just "active" or "done" — it passes through many stages. Each stage has rules about which transitions are valid.

```
                     ┌─────────────┐
                     │  requested   │  Rider tapped "Request Ride"
                     └──────┬───────┘
                            │
                            ▼
                     ┌─────────────┐
            ┌────────│  matching    │  System searching for a driver
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌──────────────────┐
            │        │ driver_assigned   │  Found a driver, waiting for response
            │        └────┬────────┬────┘
            │             │        │
            │        accept     decline/timeout
            │             │        │
            │             ▼        └──→ back to "matching"
            │        ┌─────────────┐
            │        │  accepted    │  Driver accepted the ride
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  en_route    │  Driver heading to pickup
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  arrived     │  Driver at pickup location
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │ in_progress  │  Rider in car, heading to destination
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  completed   │  Arrived at destination
            │        └─────────────┘
            │
            └──────→ ┌─────────────┐
                     │  cancelled   │  (can happen from most states)
                     └─────────────┘
```

In [ ]:
# Define valid state transitions

VALID_TRANSITIONS = {
    "requested":       ["matching", "cancelled"],
    "matching":        ["driver_assigned", "cancelled"],
    "driver_assigned": ["accepted", "matching", "cancelled"],  # matching = declined/timeout
    "accepted":        ["en_route", "cancelled"],
    "en_route":        ["arrived", "cancelled"],
    "arrived":         ["in_progress", "cancelled"],
    "in_progress":     ["completed"],  # can't cancel mid-ride
    "completed":       [],  # terminal state
    "cancelled":       [],  # terminal state
}

def can_transition(current_status, new_status):
    """Check if a state transition is valid."""
    return new_status in VALID_TRANSITIONS.get(current_status, [])


def update_ride_status(ride_id, new_status):
    """
    Transition a ride to a new status.
    Validates the transition and updates the appropriate timestamp.
    """
    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()
    
    # Get current status
    cur.execute("SELECT status FROM rides WHERE id = %s;", (ride_id,))
    row = cur.fetchone()
    if not row:
        conn.close()
        return False, f"Ride {ride_id} not found"
    
    current = row[0]
    
    # Validate transition
    if not can_transition(current, new_status):
        conn.close()
        return False, f"Invalid: {current} → {new_status}"
    
    # Update status and the appropriate timestamp
    timestamp_col = {
        "accepted": "accepted_at",
        "in_progress": "pickup_at",
        "completed": "dropoff_at",
        "cancelled": "cancelled_at",
    }.get(new_status)
    
    if timestamp_col:
        cur.execute(f"""
            UPDATE rides SET status = %s, {timestamp_col} = NOW()
            WHERE id = %s;
        """, (new_status, ride_id))
    else:
        cur.execute("""
            UPDATE rides SET status = %s WHERE id = %s;
        """, (new_status, ride_id))
    
    conn.close()
    return True, f"{current} → {new_status}"


# Demonstrate valid and invalid transitions
print("🔄 State Transition Rules:")
print()

tests = [
    ("requested", "matching",        True),
    ("requested", "completed",       False),  # can't skip to completed!
    ("matching", "driver_assigned",   True),
    ("driver_assigned", "matching",   True),   # driver declined → back to matching
    ("driver_assigned", "completed",  False),  # can't skip!
    ("in_progress", "cancelled",      False),  # can't cancel mid-ride
    ("completed", "requested",        False),  # terminal state
]

for current, target, expected in tests:
    result = can_transition(current, target)
    icon = "✅" if result == expected else "❌ BUG"
    valid = "VALID" if result else "BLOCKED"
    print(f"  {icon} {current:>16} → {target:<16} {valid}")

## 🔒 The Double-Assignment Problem

Here's a critical race condition:

```
Time    Service Instance A              Service Instance B
─────   ──────────────────              ──────────────────
  1     Ride #100 needs a driver        Ride #200 needs a driver
  2     Query: nearest driver → #5      Query: nearest driver → #5
  3     Assign driver #5 to ride #100   Assign driver #5 to ride #200
  4     ❌ Driver #5 now has TWO rides!
```

Both services found the same "best" driver and assigned them simultaneously. This violates our core requirement: **each driver should only have one active ride at a time**.

### Solution: Distributed Locks with Redis

Before assigning a driver, we **lock** them using Redis `SET key value NX EX ttl`:
- `NX` = only set if the key does **not** exist ("set if not exists")
- `EX ttl` = auto-expire after `ttl` seconds

If the SET returns True, we got the lock. If False, someone else already locked this driver.

In [ ]:
def acquire_driver_lock(driver_id, ride_id, ttl_seconds=10):
    """
    Try to lock a driver for a specific ride.
    
    Uses Redis SET NX EX:
    - NX: only succeeds if the key doesn't exist
    - EX: auto-expires after ttl_seconds
    
    Returns True if lock acquired, False if driver is already locked.
    """
    r = get_redis()
    lock_key = f"lock:driver:{driver_id}"
    
    # SET NX EX — atomic "set if not exists" with expiry
    acquired = r.set(lock_key, f"ride:{ride_id}", nx=True, ex=ttl_seconds)
    
    return bool(acquired)


def release_driver_lock(driver_id):
    """Release a driver lock (called when driver accepts/declines)."""
    r = get_redis()
    r.delete(f"lock:driver:{driver_id}")


def check_driver_lock(driver_id):
    """Check if a driver is locked and by which ride."""
    r = get_redis()
    lock_key = f"lock:driver:{driver_id}"
    ride = r.get(lock_key)
    ttl = r.ttl(lock_key)
    return ride, ttl


# Demonstrate the locking mechanism
print("🔒 Distributed Lock Demo:")
print()

# Clean up any existing locks
r = get_redis()
for key in r.keys("lock:driver:*"):
    r.delete(key)

# Scenario: Two rides both want driver #5
print("  Ride #100 tries to lock driver #5...")
result1 = acquire_driver_lock(driver_id=5, ride_id=100, ttl_seconds=10)
print(f"  → {'✅ LOCKED' if result1 else '❌ FAILED'}")

print()
print("  Ride #200 tries to lock driver #5...")
result2 = acquire_driver_lock(driver_id=5, ride_id=200, ttl_seconds=10)
print(f"  → {'✅ LOCKED' if result2 else '❌ FAILED — driver already locked!'}")

# Check the lock
ride, ttl = check_driver_lock(5)
print(f"\n  Lock status: driver #5 is locked by {ride}, expires in {ttl}s")

print()
print("💡 The second request FAILS because Redis SET NX is atomic.")
print("   Only one ride at a time can 'claim' a driver.")

In [ ]:
# Demonstrate TTL-based auto-expiry
# Scenario: Driver doesn't respond within 10 seconds

print("⏱️ TTL Lock Expiry Demo:")
print()

# Lock driver #7 with a short TTL (3 seconds for demo)
r = get_redis()
r.delete("lock:driver:7")

print("  Locking driver #7 with 3-second TTL...")
acquire_driver_lock(driver_id=7, ride_id=300, ttl_seconds=3)
ride, ttl = check_driver_lock(7)
print(f"  Lock status: {ride}, TTL={ttl}s")

# Driver doesn't respond... wait for expiry
print("  Waiting 4 seconds (driver not responding)...")
time.sleep(4)

ride, ttl = check_driver_lock(7)
print(f"  Lock status: {ride}, TTL={ttl}")
print()

# Now another ride can lock this driver
print("  Ride #400 tries to lock driver #7...")
result = acquire_driver_lock(driver_id=7, ride_id=400, ttl_seconds=10)
print(f"  → {'✅ LOCKED' if result else '❌ FAILED'}")

print()
print("💡 The lock expired automatically! No manual cleanup needed.")
print("   If a driver doesn't respond, the system moves to the next driver.")

# Cleanup
r.delete("lock:driver:7")

## 🚗 Complete Ride Flow: Request to Completion

Let's put everything together and simulate a complete ride lifecycle:

1. Rider requests a ride
2. System finds nearby drivers
3. System locks the best driver and sends the request
4. If driver declines or times out → try the next driver
5. Driver accepts → ride proceeds through remaining states

In [ ]:
# Load drivers into Redis for matching
r = get_redis()
conn = get_db()
cur = conn.cursor()

r.delete("drivers:locations", "drivers:available")
for key in r.keys("lock:driver:*"):
    r.delete(key)

cur.execute("""
    SELECT d.id, d.status,
           ST_X(dl.location::geometry) AS lng,
           ST_Y(dl.location::geometry) AS lat
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")
for row in cur.fetchall():
    did, status, lng, lat = row
    r.geoadd("drivers:locations", (lng, lat, f"driver:{did}"))
    if status == "available":
        r.sadd("drivers:available", f"driver:{did}")

conn.close()
print(f"✅ Loaded drivers: {r.scard('drivers:available')} available")

In [ ]:
def request_ride(rider_id, pickup_lng, pickup_lat, dropoff_lng, dropoff_lat):
    """
    Complete ride request flow:
    1. Create the ride in the database
    2. Find nearby drivers
    3. Try to lock and assign each one until success
    """
    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()
    r = get_redis()
    
    # Step 1: Create the ride
    cur.execute("""
        INSERT INTO rides (rider_id, status, pickup_location, dropoff_location)
        VALUES (%s, 'requested',
                ST_SetSRID(ST_MakePoint(%s, %s), 4326),
                ST_SetSRID(ST_MakePoint(%s, %s), 4326))
        RETURNING id;
    """, (rider_id, pickup_lng, pickup_lat, dropoff_lng, dropoff_lat))
    ride_id = cur.fetchone()[0]
    print(f"  📱 Ride #{ride_id} created (status: requested)")
    
    # Step 2: Transition to matching
    cur.execute("UPDATE rides SET status = 'matching' WHERE id = %s;", (ride_id,))
    print(f"  🔍 Status → matching")
    
    # Step 3: Find nearby available drivers
    nearby = r.geosearch(
        name="drivers:locations",
        longitude=pickup_lng, latitude=pickup_lat,
        radius=5, unit="km",
        withdist=True, sort="ASC", count=10
    )
    available = r.smembers("drivers:available")
    candidates = [(m, d) for m, d in nearby if m in available]
    
    print(f"  📋 Found {len(candidates)} nearby available drivers")
    
    # Step 4: Try each driver
    for driver_key, distance in candidates:
        driver_id = int(driver_key.split(":")[1])
        
        # Try to lock this driver
        locked = acquire_driver_lock(driver_id, ride_id, ttl_seconds=10)
        
        if not locked:
            print(f"  ❌ {driver_key} is locked (assigned to another ride)")
            continue
        
        print(f"  🔒 Locked {driver_key} ({float(distance):.2f} km away)")
        
        # Update ride to driver_assigned
        cur.execute("""
            UPDATE rides SET status = 'driver_assigned', driver_id = %s
            WHERE id = %s;
        """, (driver_id, ride_id))
        print(f"  📤 Status → driver_assigned (sent request to {driver_key})")
        
        # Simulate driver decision (80% accept for demo)
        accepts = random.random() < 0.8
        
        if accepts:
            print(f"  ✅ {driver_key} ACCEPTED!")
            cur.execute("""
                UPDATE rides SET status = 'accepted', accepted_at = NOW()
                WHERE id = %s;
            """, (ride_id,))
            
            # Remove from available set
            r.srem("drivers:available", driver_key)
            # Release the lock (driver is now "busy", not just "locked")
            release_driver_lock(driver_id)
            
            conn.close()
            return ride_id, driver_id
        else:
            print(f"  ❌ {driver_key} DECLINED, trying next...")
            release_driver_lock(driver_id)
            # Back to matching
            cur.execute("UPDATE rides SET status = 'matching', driver_id = NULL WHERE id = %s;", (ride_id,))
    
    # No driver found
    cur.execute("UPDATE rides SET status = 'cancelled', cancelled_at = NOW() WHERE id = %s;", (ride_id,))
    print(f"  ❌ No drivers available — ride cancelled")
    conn.close()
    return ride_id, None


# Request a ride!
print("=" * 60)
print("🚗 Requesting a ride: Downtown SF → Mission District")
print("=" * 60)

ride_id, driver_id = request_ride(
    rider_id=1,
    pickup_lng=-122.4194, pickup_lat=37.7749,   # Downtown
    dropoff_lng=-122.4103, dropoff_lat=37.7627   # Mission
)

In [ ]:
# Complete the ride lifecycle (if a driver was assigned)

if driver_id:
    print(f"\n🚗 Completing ride #{ride_id} with driver #{driver_id}")
    print("=" * 50)
    
    steps = [
        ("en_route",    "🚗 Driver heading to pickup..."),
        ("arrived",     "📍 Driver arrived at pickup!"),
        ("in_progress", "🛣️  Rider picked up, heading to destination..."),
        ("completed",   "🏁 Arrived! Ride complete."),
    ]
    
    for new_status, message in steps:
        time.sleep(0.5)  # small delay for readability
        success, detail = update_ride_status(ride_id, new_status)
        icon = "✅" if success else "❌"
        print(f"  {icon} {message} ({detail})")
    
    # Mark driver as available again
    r = get_redis()
    r.sadd("drivers:available", f"driver:{driver_id}")
    print(f"\n  🟢 Driver #{driver_id} is available again")
    
    # Show the final ride record
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT id, rider_id, driver_id, status,
               requested_at, accepted_at, pickup_at, dropoff_at
        FROM rides WHERE id = %s;
    """, (ride_id,))
    row = cur.fetchone()
    conn.close()
    
    print(f"\n  📋 Final Ride Record:")
    print(f"     Ride ID:      {row[0]}")
    print(f"     Rider:        {row[1]}")
    print(f"     Driver:       {row[2]}")
    print(f"     Status:       {row[3]}")
    print(f"     Requested:    {row[4]}")
    print(f"     Accepted:     {row[5]}")
    print(f"     Picked up:    {row[6]}")
    print(f"     Dropped off:  {row[7]}")
else:
    print("\n❌ No driver was assigned. Try running the cell again!")

## 🏎️ Concurrent Ride Requests (Race Condition Test)

Let's prove that our locking prevents double-assignment by requesting two rides simultaneously that both want the same driver.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Reset driver availability
r = get_redis()
for key in r.keys("lock:driver:*"):
    r.delete(key)

# Pre-lock all drivers except #1 — force both rides to compete for driver #1
for did in range(2, 11):
    r.srem("drivers:available", f"driver:{did}")

# Only driver #1 is available
r.sadd("drivers:available", "driver:1")
print(f"Available drivers: {r.smembers('drivers:available')}")
print()

# Both rides try to get driver #1
results = {}

def try_lock(ride_id):
    locked = acquire_driver_lock(driver_id=1, ride_id=ride_id, ttl_seconds=10)
    return ride_id, locked

print("🏎️ Two rides competing for driver #1:")
print()

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(try_lock, rid) for rid in [500, 501]]
    for f in futures:
        ride_id, locked = f.result()
        status = "✅ GOT THE LOCK" if locked else "❌ BLOCKED"
        print(f"  Ride #{ride_id}: {status}")

# Check who holds the lock
ride, ttl = check_driver_lock(1)
print(f"\n  🔒 Driver #1 is locked by: {ride}")
print()
print("✅ Only ONE ride got the driver — no double-assignment!")

# Cleanup
release_driver_lock(1)
for did in range(1, 11):
    r.sadd("drivers:available", f"driver:{did}")

## 🧹 Cleanup

In [ ]:
r = get_redis()
for key in r.keys("lock:driver:*"):
    r.delete(key)
r.delete("drivers:locations", "drivers:available")
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **State machines** enforce valid ride transitions — you can't skip from "requested" to "completed"
2. **Distributed locks** (Redis SET NX EX) prevent double-assignment of drivers
3. **TTL on locks** means unresponsive drivers don't block the system — the lock auto-expires
4. **The matching loop** tries drivers in distance order, skipping any that are already locked
5. **Concurrency safety** — even with multiple service instances, only one ride can claim a driver

### How This Fits in a System Design Interview

The ride lifecycle question tests:
- **State management** — can you model complex workflows as state machines?
- **Consistency** — how do you prevent race conditions in a distributed system?
- **Failure handling** — what happens if a driver/service crashes mid-assignment?
- **Lock strategies** — application locks vs. database locks vs. distributed locks (Redis)

The progression:
- ❌ Bad: application-level locks → no coordination between instances
- ✅ Good: database locks → coordinated but locks stuck if service crashes
- ✅ Great: Redis distributed locks with TTL → atomic, coordinated, auto-expiring

### 🎓 What We Covered in This Lab Series

| Notebook | Concept | Key Technology |
|----------|---------|----------------|
| 1. Geospatial Matching | Finding nearby drivers | PostGIS + Redis GEOSEARCH |
| 2. Real-Time Tracking | Handling millions of location updates | Redis GEOADD + staleness cleanup |
| 3. Surge Pricing | Dynamic pricing from supply/demand | Zone counting + multiplier formula |
| 4. Trip Lifecycle | Ride state machine + preventing race conditions | State transitions + Redis distributed locks |